# E011 spline4 inference
Attach the E011 input dataset and the ROGII competition data, then run all cells. The notebook writes the submission and a small run receipt to `/kaggle/working`.


In [ ]:
CONTRACT = {
  "BUNDLE_FILENAME": "rogii-e011-deployment-inputs-v1.zip.bin",
  "DATASET_REF": "ashok205/rogii-e011-deployment-inputs",
  "DATASET_VERSION": 1,
  "RUN_RECEIPT_FILENAME": "e011-run-receipt.json",
  "SUBMISSION_FILENAME": "submission.csv",
  "WELL_DIAGNOSTICS_FILENAME": "e011-well-predictions.json"
}


In [ ]:
import datetime as _dt
import importlib
import json
import shutil
import sys
import traceback
import zipfile
from pathlib import Path, PurePosixPath

WORKING = Path("/kaggle/working")
RUNTIME_ROOT = WORKING / "rogii-e011-runtime"
RUN_RECEIPT_PATH = WORKING / CONTRACT["RUN_RECEIPT_FILENAME"]
SUBMISSION_PATH = WORKING / CONTRACT["SUBMISSION_FILENAME"]
WELL_DIAGNOSTICS_PATH = WORKING / CONTRACT["WELL_DIAGNOSTICS_FILENAME"]
STARTED_AT = _dt.datetime.now(_dt.timezone.utc)


def _safe_member(name: str) -> str:
    raw = str(name).replace("\\", "/")
    value = PurePosixPath(raw)
    if value.is_absolute() or not raw or any(part in {"", ".", ".."} for part in value.parts):
        raise RuntimeError(f"unsafe archive member: {name!r}")
    return value.as_posix()


def _write_failure(stage: str, exc: BaseException) -> None:
    payload = {
        "status": "FAILED",
        "stage": stage,
        "error_type": type(exc).__name__,
        "error": str(exc),
        "failed_at_utc": _dt.datetime.now(_dt.timezone.utc).isoformat(),
        "traceback": traceback.format_exc(),
        "kaggle_submission_made": False,
    }
    RUN_RECEIPT_PATH.write_text(
        json.dumps(payload, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )


try:
    bundle_matches = sorted(Path("/kaggle/input").rglob(CONTRACT["BUNDLE_FILENAME"]))
    if len(bundle_matches) != 1:
        raise RuntimeError({"bundle_matches": [str(path) for path in bundle_matches]})
    bundle_path = bundle_matches[0]

    if RUNTIME_ROOT.exists():
        shutil.rmtree(RUNTIME_ROOT)
    RUNTIME_ROOT.mkdir(parents=True)
    with zipfile.ZipFile(bundle_path) as archive:
        names = archive.namelist()
        if len(names) != len(set(names)):
            raise RuntimeError("duplicate archive members")
        for name in names:
            _safe_member(name)
        archive.extractall(RUNTIME_ROOT)

    for module_name in tuple(sys.modules):
        if module_name == "rogii_validation" or module_name.startswith("rogii_validation."):
            del sys.modules[module_name]
    importlib.invalidate_caches()
    sys.path.insert(0, str(RUNTIME_ROOT / "src"))

    from rogii_validation.e011_inference import load_e011_model, write_e011_submission

    preferred_roots = (
        Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction"),
        Path("/kaggle/input/rogii-wellbore-geology-prediction"),
    )
    competition_root = next(
        (
            root
            for root in preferred_roots
            if (root / "test").is_dir() and (root / "sample_submission.csv").is_file()
        ),
        None,
    )
    if competition_root is None:
        candidates = sorted(
            sample.parent
            for sample in Path("/kaggle/input").rglob("sample_submission.csv")
            if (sample.parent / "test").is_dir()
        )
        if len(candidates) != 1:
            raise RuntimeError({"competition_roots_found": [str(root) for root in candidates]})
        competition_root = candidates[0]

    model_path = RUNTIME_ROOT / "experiments/E011/deployment/model.json"
    if not model_path.is_file():
        raise FileNotFoundError(model_path)

    model = load_e011_model(model_path)
    result = write_e011_submission(
        model,
        competition_root / "test",
        competition_root / "sample_submission.csv",
        SUBMISSION_PATH,
    )
    WELL_DIAGNOSTICS_PATH.write_text(
        json.dumps(result["well_predictions"], indent=2, sort_keys=True, allow_nan=False) + "\n",
        encoding="utf-8",
    )
    ended_at = _dt.datetime.now(_dt.timezone.utc)
    receipt = {
        "status": "COMPLETE",
        "experiment_id": "E011",
        "dataset": {
            "ref": CONTRACT["DATASET_REF"],
            "version": CONTRACT["DATASET_VERSION"],
        },
        "competition_root": str(competition_root),
        "started_at_utc": STARTED_AT.isoformat(),
        "ended_at_utc": ended_at.isoformat(),
        "wall_seconds": (ended_at - STARTED_AT).total_seconds(),
        "prediction": {key: value for key, value in result.items() if key != "well_predictions"},
        "outputs": [
            SUBMISSION_PATH.name,
            RUN_RECEIPT_PATH.name,
            WELL_DIAGNOSTICS_PATH.name,
        ],
        "submission_file_created": True,
        "kaggle_submission_made": False,
    }
    RUN_RECEIPT_PATH.write_text(
        json.dumps(receipt, indent=2, sort_keys=True, allow_nan=False) + "\n",
        encoding="utf-8",
    )
    print(json.dumps(receipt, indent=2, sort_keys=True))
except Exception as exc:
    _write_failure("run", exc)
    raise
